# YgoMaster-PackBuilder — Build Validation Tests

Test offline di `validate_ygomaster_build()`.

Copre build valida, file mancanti, JSON invalido, root strutturali errate, `PerPackRarities`, riferimenti Shop↔Odds, mismatch di contenuto, `.tmp`, file estranei consentiti ed errore nel calcolo dell'expected export.


In [1]:
from copy import deepcopy
from pathlib import Path
from tempfile import TemporaryDirectory
import json

from ygomaster_packbuilder.domain import (
    Card, OpeningPolicy, PackDefinition, PackType, ProjectDefinition,
    Rarity, RarityDistribution, SourcePackRef, SourceProvider,
    TenPackGuarantee, UltraRarePity,
)
from ygomaster_packbuilder.exporters.ygomaster import write_ygomaster_build
from ygomaster_packbuilder.validation import validate_ygomaster_build

def make_policy():
    return OpeningPolicy(
        key="configuration-a",
        normal_slot_distribution=RarityDistribution(55.0,35.0,7.5,2.5),
        ten_pack_guarantee=TenPackGuarantee(
            True, RarityDistribution(0.0,0.0,80.0,20.0)
        ),
        ultra_rare_pity=UltraRarePity(
            True, RarityDistribution(0.0,0.0,0.0,100.0)
        ),
    )

def make_card(name, ydk_id, md_id, rarity):
    return Card(
        name=name, ydk_id=ydk_id, source_rarity=rarity.value,
        master_duel_id=md_id, rarity=rarity
    )

def make_pack(key, pack_type, booster_size, cards):
    expected = {
        PackType.PERSONAL_MAIN: 100,
        PackType.PERSONAL_MINI: 20,
        PackType.SHARED_STAPLES: 200,
    }[pack_type]
    return PackDefinition(
        key=key, name=key, description=f"{key} description",
        type=pack_type,
        source=SourcePackRef(
            provider=SourceProvider.YGOPRODECK,
            external_id=key
        ),
        expected_card_count=expected,
        booster_size=booster_size,
        opening_policy_key="configuration-a",
        cards=cards,
    )

project = ProjectDefinition(
    name="Build Validation Test",
    packs=[
        make_pack(
            "main-pack", PackType.PERSONAL_MAIN, 7,
            [make_card("N",1,1001,Rarity.N), make_card("UR",2,1002,Rarity.UR)]
        ),
        make_pack(
            "mini-pack", PackType.PERSONAL_MINI, 3,
            [make_card("R",3,1003,Rarity.R), make_card("SR",4,1004,Rarity.SR)]
        ),
    ],
    opening_policies=[make_policy()],
)

pack_ids = {
    "main-pack": 9_000_000,
    "mini-pack": 9_000_001,
}

def codes(report):
    return [issue.code for issue in report.errors]

def assert_code(report, expected):
    actual = codes(report)
    assert expected in actual, f"Expected {expected!r}, got {actual}"
    print(expected, "OK")

print("Fixtures ready")

Fixtures ready


## 1. Build valida

In [2]:
with TemporaryDirectory() as tmp:
    build_dir = Path(tmp) / "build"
    write_ygomaster_build(project, pack_ids, build_dir)
    report = validate_ygomaster_build(project, pack_ids, build_dir)
    assert report.is_valid
    assert report.errors == []
print("Valid build accepted")

Valid build accepted


## 2. File estranei consentiti

In [3]:
with TemporaryDirectory() as tmp:
    build_dir = Path(tmp) / "build"
    write_ygomaster_build(project, pack_ids, build_dir)
    (build_dir / "other-mod.json").write_text('{"hello":"world"}', encoding="utf-8")
    report = validate_ygomaster_build(project, pack_ids, build_dir)
    assert report.is_valid
print("Unrelated files ignored")

Unrelated files ignored


## 3. Build assente

In [4]:
with TemporaryDirectory() as tmp:
    build_dir = Path(tmp) / "missing-build"
    report = validate_ygomaster_build(project, pack_ids, build_dir)
    assert codes(report).count("build-file-missing") == 3
print("Missing build detected")

Missing build detected


## 4. JSON invalido

In [5]:
with TemporaryDirectory() as tmp:
    build_dir = Path(tmp) / "build"
    artifacts = write_ygomaster_build(project, pack_ids, build_dir)
    artifacts.shop.write_text("{ definitely not json", encoding="utf-8")
    report = validate_ygomaster_build(project, pack_ids, build_dir)
    assert_code(report, "build-invalid-json")

build-invalid-json OK


## 5. Shop root / PackShop invalidi

In [6]:
with TemporaryDirectory() as tmp:
    build_dir = Path(tmp) / "build"
    artifacts = write_ygomaster_build(project, pack_ids, build_dir)
    artifacts.shop.write_text("[]", encoding="utf-8")
    report = validate_ygomaster_build(project, pack_ids, build_dir)
    assert_code(report, "build-invalid-shop-root")
    assert_code(report, "build-shop-content-mismatch")

with TemporaryDirectory() as tmp:
    build_dir = Path(tmp) / "build"
    artifacts = write_ygomaster_build(project, pack_ids, build_dir)
    artifacts.shop.write_text(json.dumps({"Wrong": {}}), encoding="utf-8")
    report = validate_ygomaster_build(project, pack_ids, build_dir)
    assert_code(report, "build-invalid-pack-shop")

build-invalid-shop-root OK
build-shop-content-mismatch OK
build-invalid-pack-shop OK


## 6. Odds root invalida

In [7]:
with TemporaryDirectory() as tmp:
    build_dir = Path(tmp) / "build"
    artifacts = write_ygomaster_build(project, pack_ids, build_dir)
    artifacts.shop_pack_odds.write_text("{}", encoding="utf-8")
    report = validate_ygomaster_build(project, pack_ids, build_dir)
    assert_code(report, "build-invalid-odds-root")
    assert_code(report, "build-odds-content-mismatch")

build-invalid-odds-root OK
build-odds-content-mismatch OK


## 7. Settings patch invalida

In [8]:
with TemporaryDirectory() as tmp:
    build_dir = Path(tmp) / "build"
    artifacts = write_ygomaster_build(project, pack_ids, build_dir)
    artifacts.shop_settings_patch.write_text("[]", encoding="utf-8")
    report = validate_ygomaster_build(project, pack_ids, build_dir)
    assert_code(report, "build-invalid-settings-patch-root")
    assert_code(report, "build-settings-patch-mismatch")

with TemporaryDirectory() as tmp:
    build_dir = Path(tmp) / "build"
    artifacts = write_ygomaster_build(project, pack_ids, build_dir)
    artifacts.shop_settings_patch.write_text(
        json.dumps({"PerPackRarities": False}),
        encoding="utf-8"
    )
    report = validate_ygomaster_build(project, pack_ids, build_dir)
    assert_code(report, "build-per-pack-rarities-disabled")
    assert_code(report, "build-settings-patch-mismatch")

build-invalid-settings-patch-root OK
build-settings-patch-mismatch OK
build-per-pack-rarities-disabled OK
build-settings-patch-mismatch OK


## 8. Odds entry / packShopIds invalidi

In [9]:
with TemporaryDirectory() as tmp:
    build_dir = Path(tmp) / "build"
    artifacts = write_ygomaster_build(project, pack_ids, build_dir)
    odds = json.loads(artifacts.shop_pack_odds.read_text(encoding="utf-8"))
    odds[0] = 123
    artifacts.shop_pack_odds.write_text(json.dumps(odds), encoding="utf-8")
    report = validate_ygomaster_build(project, pack_ids, build_dir)
    assert_code(report, "build-invalid-odds-entry")
    assert_code(report, "build-shop-odds-reference-mismatch")

with TemporaryDirectory() as tmp:
    build_dir = Path(tmp) / "build"
    artifacts = write_ygomaster_build(project, pack_ids, build_dir)
    odds = json.loads(artifacts.shop_pack_odds.read_text(encoding="utf-8"))
    odds[0]["packShopIds"] = "not-a-list"
    artifacts.shop_pack_odds.write_text(json.dumps(odds), encoding="utf-8")
    report = validate_ygomaster_build(project, pack_ids, build_dir)
    assert_code(report, "build-invalid-odds-pack-ids")
    assert_code(report, "build-shop-odds-reference-mismatch")

build-invalid-odds-entry OK
build-shop-odds-reference-mismatch OK
build-invalid-odds-pack-ids OK
build-shop-odds-reference-mismatch OK


## 9. Shop ↔ Odds mismatch

In [10]:
with TemporaryDirectory() as tmp:
    build_dir = Path(tmp) / "build"
    artifacts = write_ygomaster_build(project, pack_ids, build_dir)
    odds = json.loads(artifacts.shop_pack_odds.read_text(encoding="utf-8"))
    odds[0]["packShopIds"] = [19999999]
    artifacts.shop_pack_odds.write_text(json.dumps(odds), encoding="utf-8")
    report = validate_ygomaster_build(project, pack_ids, build_dir)
    assert_code(report, "build-shop-odds-reference-mismatch")

build-shop-odds-reference-mismatch OK


## 10. Content mismatch

In [11]:
with TemporaryDirectory() as tmp:
    build_dir = Path(tmp) / "build"
    artifacts = write_ygomaster_build(project, pack_ids, build_dir)
    shop = json.loads(artifacts.shop.read_text(encoding="utf-8"))
    next(iter(shop["PackShop"].values()))["price"] = 999
    artifacts.shop.write_text(json.dumps(shop), encoding="utf-8")
    report = validate_ygomaster_build(project, pack_ids, build_dir)
    assert_code(report, "build-shop-content-mismatch")

with TemporaryDirectory() as tmp:
    build_dir = Path(tmp) / "build"
    artifacts = write_ygomaster_build(project, pack_ids, build_dir)
    odds = json.loads(artifacts.shop_pack_odds.read_text(encoding="utf-8"))
    odds[0]["gachaType"] = 999
    artifacts.shop_pack_odds.write_text(json.dumps(odds), encoding="utf-8")
    report = validate_ygomaster_build(project, pack_ids, build_dir)
    assert_code(report, "build-odds-content-mismatch")

with TemporaryDirectory() as tmp:
    build_dir = Path(tmp) / "build"
    artifacts = write_ygomaster_build(project, pack_ids, build_dir)
    artifacts.shop_settings_patch.write_text(
        json.dumps({"PerPackRarities": True, "Unexpected": 123}),
        encoding="utf-8"
    )
    report = validate_ygomaster_build(project, pack_ids, build_dir)
    assert_code(report, "build-settings-patch-mismatch")

build-shop-content-mismatch OK
build-odds-content-mismatch OK
build-settings-patch-mismatch OK


## 11. File `.tmp` residuo

In [12]:
with TemporaryDirectory() as tmp:
    build_dir = Path(tmp) / "build"
    write_ygomaster_build(project, pack_ids, build_dir)
    (build_dir / "Shop" / "interrupted.tmp").write_text("partial", encoding="utf-8")
    report = validate_ygomaster_build(project, pack_ids, build_dir)
    assert_code(report, "build-temp-file-present")

build-temp-file-present OK


## 12. Più errori contemporaneamente

In [13]:
with TemporaryDirectory() as tmp:
    build_dir = Path(tmp) / "build"
    artifacts = write_ygomaster_build(project, pack_ids, build_dir)
    artifacts.shop.unlink()
    artifacts.shop_settings_patch.write_text(
        json.dumps({"PerPackRarities": False}),
        encoding="utf-8"
    )
    (build_dir / "ShopPackOdds" / "stale.tmp").write_text("partial", encoding="utf-8")
    report = validate_ygomaster_build(project, pack_ids, build_dir)
    actual = set(codes(report))
    assert {
        "build-file-missing",
        "build-per-pack-rarities-disabled",
        "build-settings-patch-mismatch",
        "build-temp-file-present",
    }.issubset(actual)
    print(sorted(actual))

['build-file-missing', 'build-per-pack-rarities-disabled', 'build-settings-patch-mismatch', 'build-temp-file-present']


## 13. Expected export non calcolabile

In [14]:
with TemporaryDirectory() as tmp:
    build_dir = Path(tmp) / "build"
    write_ygomaster_build(project, pack_ids, build_dir)
    bad_ids = {"main-pack": 9_000_000}
    report = validate_ygomaster_build(project, bad_ids, build_dir)
    assert_code(report, "build-expected-data-error")

build-expected-data-error OK
